# XCalib Tutorial: Homography from Pairs

Welcome. This notebook takes a raw data delivery to the homography that places every LiDAR object in the camera image: `[u, v, 1] ~ H @ [position_x, position_y, 1]`. No extrinsic is needed: the homography is fitted to pairs, camera boxes and LiDAR boxes that show the same object.

The pairs come from three sources, and each step says which one it uses:

- pairs you already have, in a folder saved earlier;
- pairs you confirm yourself in the pairing window;
- pairs that a model predicts once it has been trained on your pairs. You check them in the pairing window before they are saved.

The notebook has three parts, and you run only the ones you need (always after the first code cell):

- **Part 1, base model (steps 1 to 4):** the base data is imported, you pair it, and a base model is trained on the pairs.
- **Part 2, refine (steps 5 to 8):** the refine data is imported (other raw data than the base data), a base model suggests its pairs, you correct them, the model is fine-tuned on them and pairs the remaining frames. The base model is the one of step 4 or one you already have, so Part 1 is not needed for it.
- **Homography (step 9):** written at the end of Part 2, from the saved pairs of the refine data, and of the base data too when Part 1 ran in the same session. It needs the refine part, not Part 1.

Please run the cells from top to bottom, one at a time: select the code cell, press **Shift + Enter**, and wait until it has finished (the `[*]` on the left turns into a number). A dialog opens whenever a folder or a file has to be chosen; if you do not see it, please check the taskbar. Please keep the JupyterLab debugger (the bug icon at the top right) switched off: dialogs and windows cannot open while it is on. A message that starts with ⚠ explains what is needed. To stop a training, or the model's pairing in step 8, while it runs, press the ■ button in the toolbar: the training or the pairing stops with the cell. A cell that waits for a dialog or a pairing window ends when you close that window; the ■ button never closes a pairing window, so nothing you decided there is lost.

To open this notebook, run `cd demo`, `pixi install` and `pixi run notebook-homography` from the repository folder. The task also registers the **XCalib demo (.pixi)** kernel of this environment: if JupyterLab asks for a kernel, please select it at the top right. Then run the next cell. The two numbers are the training epochs of the base model and of its fine-tuning, and you can change them: 30 and 10 take about an hour on a computer with an NVIDIA GPU and several hours without one; 1 and 1 only check that everything runs. If you change a number later, edit it and run this cell again: that starts a new session, in which nothing imported or saved is lost, but the scenes and models chosen so far are forgotten. Steps 4 and 7 say which cells to run again then.

In [ ]:
from xcalib.gui.customer import Workflow

flow = Workflow(base_epochs=30, refine_epochs=10)

# Part 1: base model

Steps 1 to 4 import the base data, pair it and train a base model on the pairs. If you already have a base model, please skip this part and go on to Part 2: the refine part and the homography do not need it.

## 1. Import the base data

Run the next cell and choose the raw data folder of the base data in the dialog. Please make sure that the delivery is already calibrated: the images are undistorted, and the camera detections were made on those images.

The camera intrinsic is read from `undistortion_parameters.json` in the data folder. If the file is kept somewhere else, a second dialog asks for it: please choose the file that holds the camera matrix of the undistorted images.

The import copies the synchronized images and point clouds, all frames of the folder, into one scene, the **base** scene, and writes the camera and LiDAR detections as draft boxes. The scene is written into `datasets/raw_workflow/<name of the raw data folder>` (in that name, each run of characters other than letters, digits, `-` and `_` becomes a single `_`). Three check images are shown at the end: the boxes should enclose the vehicles.

If this folder has been imported before, running the cell again does not copy the data again: it shows the three check images once more and uses the scene that is there. A base scene imported before can be chosen in the same dialog too (its folder name ends with `_base`).

In [ ]:
flow.import_raw("base")

## Optional: look at the base scene in SUSTechPOINTS

Run the next cell to open the base scene in SUSTechPOINTS, a 3D annotation tool, in your web browser. Every frame shows its point cloud with the LiDAR boxes written in step 1 and their LiDAR ids; with the mouse over the large view, the left and right arrow keys go to the previous and the next frame (PageUp and PageDown too). To see the camera image, choose the camera (for example `cam_sync`) in the **Camera** menu at the top; it shows no boxes, because this workflow has no extrinsic. The page is a good place to check the LiDAR boxes, to fit them to their points, and to look up the LiDAR id of an object before you pair it.

To fine-tune a LiDAR box, click its name (or one of its lines): three views of it open on the left, from above, from the side and from behind. There, drag a border or a corner (take it just inside the line), or drag the middle to move the box; a double click on a border pulls it in to the points inside the box, a double click in the middle pulls in every border, so drag a border out first where the box cuts the object. With the mouse over one of these views, W/A/S/D and the arrow keys move the box; over the view from above, Q/E turn it. The page saves every change into the scene at once. Please do not delete a box (Delete, or Ctrl+D over these views) and do not change its id: the saved pairs name LiDAR boxes by their ids. Move the mouse off the three views before you change the frame.

The page stays available while this notebook runs (the first start takes about half a minute), and running the cell again opens it again. SUSTechPOINTS is a separate program with its own files in the `datasets` folder (see the SUSTechPOINTS section of `docs/review-gui.md`); when one of them is missing, the cell says which. If you do not need this view, please skip the cell.

In [ ]:
flow.view_sustech("base")

## Optional: use pairs you already have

If the frames of this delivery were paired before (in an earlier session, for example), run the next cell and choose that folder in the dialog: the scene folder in which the pairs were saved, its `label_fusion/camera/<camera>` folder, or a folder that holds the saved label files themselves (one `<frame>.json` per frame; the dialog shows folders only, so such a folder looks empty in it). Every pair and every "no LiDAR box" mark is copied onto the same camera box of this scene, so nothing has to be clicked again. Decisions this scene already holds are kept, and a pair is not copied onto a LiDAR box that another camera box of the frame holds already, so running the cell again changes nothing you decided.

Below the cell you see how many pairs were copied and where in the image they lie. If you have no pairs yet, please skip this cell.

In [ ]:
flow.import_pairs("base")

## 2. Confirm pairs

Run the next cell. In the window, the camera image is on the left and the LiDAR boxes seen from above are on the right. Click a camera box in the image, then click its LiDAR box in the list below (or **— none —** when the object has no LiDAR box in this frame). When a frame is done, click **Apply this frame**; the arrow keys go to the next frame. Only applied decisions are saved: what is not applied when the window closes is lost (**Apply all** saves the decisions of every frame at once). Pairs copied in the step above are shown here as well: this is where you check them, correct them and add more.

In a frame where you save a pair, please decide every camera box: give it its LiDAR box, or **— none —**. The training in step 4 counts a camera box left without a decision in such a frame as "no LiDAR box", so a pair left open would teach the model that the object has none. After the window closes, the cell lists such frames if there are any.

The homography is fitted to these pairs and the model learns from them, so every saved pair should be right. Please confirm at least 12 pairs, and more if you can, on objects on the left, in the middle and on the right of the image, near and far. Pairs whose camera box is cut by the image edge, or wider than 8% of the image (an object close to the camera), take no part in the homography, and a parked object counts only once however many frames it is paired in: please make sure that enough of the pairs are whole boxes of moving objects that are not close to the camera, at different places in the image. Close the window when you are done.

In [ ]:
flow.pair_for_homography("base")

## 3. View the pairs

Run the next cell to see what is saved: the number of pairs, how the pairs that the homography can use are spread over the image (left, middle, right), and three check images. In a check image every pair has one colour, on the camera box in the image (left) and on its LiDAR box seen from above (right), with the LiDAR id written at both. A gray camera box is marked "no LiDAR box"; a white one has no decision yet, and in a frame with a saved pair the training counts it as "no LiDAR box" too (step 2 is where to decide it).

You can run this cell again at any time. To change a pair, go back to step 2.

In [ ]:
flow.view_pairs("base")

## 4. Train the pairing model

Run the next cell. The frames with saved pairs are exported as training data, HDF5 files in a new folder `runs/customer-base-<date>-<time>` (`train.h5`, `validation.h5`, and `split.json`, which lists the frames of each), and a model is trained on them for the number of epochs set in the first cell. An exported frame holds its pairs, its camera boxes marked "no LiDAR box" and its other LiDAR boxes; a camera box without a decision in such a frame counts as "no LiDAR box" too, and the cell says how many there are. The line below the cell shows the current epoch, the last validation result and the time left. Please keep the computer running, and do not let it go to sleep, until the cell has finished.

The model is saved in the `model` folder of that run; it is what pairs new frames for you in the steps that follow. If the notebook is restarted later, please run the first cell again: the next steps then ask in a dialog for what they need, such as a scene or this model folder.

To train again with other epochs, change the first number in the first cell and run that cell again, then step 1 with the same folder (nothing is copied again, and your pairs stay) and this cell. The new model is written to a new folder; the earlier one stays where it is.

In [ ]:
flow.train("base")

# Part 2: refine on new data

Steps 5 to 8 need a base model: the one trained in step 4, or one you already have (the optional cell "use a base model you already have" after step 5). They need nothing else from Part 1. Pairs you already have for the refine data can be copied in by the optional cell "use pairs you already have for the refine data", before step 6.

## 5. Import the refine data

Run the next cell and choose the raw data folder of the refine data in the dialog: the delivery the model is to be refined on. It should be other raw data than the base data of step 1: frames the base model has not seen, from another recording or from another part of the same one. Data that also holds the base model's frames, such as the whole recording, works as well: every step runs and step 9 uses every saved pair, but the fine-tuning then learns little that is new. It is imported as in step 1: all of its frames become one scene, the **refine** scene, written into `datasets/raw_workflow/<name of the raw data folder>/refine`, and three check images are shown.

The camera intrinsic is read from `undistortion_parameters.json` in the data folder. When the folder has none, the intrinsic of the base scene of step 1 is used (the same camera); without a base scene, a second dialog asks for the file.

If this folder has been imported before, the data is not copied again. A refine scene imported before can be chosen in the same dialog too (its folder name ends with `_refine`).

In [ ]:
flow.import_raw("refine")

## Optional: look at the refine scene in SUSTechPOINTS

Run the next cell to open the refine scene in SUSTechPOINTS, as the optional cell "look at the base scene in SUSTechPOINTS" of Part 1 does for the base scene. When a base scene is chosen in this session too, the page holds both scenes: choose a scene in the scene list at the top and then a frame in the frame list beside it, or open the scene's own address, which the cell prints. A page that showed the base scene alone is then closed and starts again at a new address, so please go on in the tab that opens. Boxes fine-tuned there are saved into their own scene, as in Part 1; please do not delete a box or change its id. If you do not need this view, please skip the cell.

In [ ]:
flow.view_sustech("refine")

## Optional: use a base model you already have

If a base model was trained before (in an earlier session of this notebook, or delivered to you), run the next cell and choose its folder in the dialog: the `model` folder of a training run, which holds `crlite.pth` and `crlite.yaml` (choosing the folder of the run is enough). Steps 6 and 7 then use this model, so steps 1 to 4 are not needed.

Please skip this cell when you trained the base model in step 4 since the first cell was last run: that model is used.

In [ ]:
flow.choose_model("base")

## Optional: use pairs you already have for the refine data

If the frames of the refine data were paired before, run the next cell and choose that folder in the dialog, as in the optional cell "use pairs you already have" of Part 1: the scene folder in which the pairs were saved, its `label_fusion/camera/<camera>` folder, or a folder that holds the saved label files themselves (one `<frame>.json` per frame; the dialog shows folders only, so such a folder looks empty in it). Every pair and every "no LiDAR box" mark is copied onto the same camera box of the refine scene, and decisions the scene already holds are kept. Step 6 then shows the copied pairs for checking, and the model suggests pairs only for the camera boxes without a decision; steps 7 and 9 use the copied pairs as well.

Below the cell you see how many pairs were copied and where in the image they lie. If you have no pairs for this data, please skip this cell.

In [ ]:
flow.import_pairs("refine")

## 6. Confirm the model's suggested pairs

Run the next cell. The base model (trained in step 4, or chosen in the optional cell "use a base model you already have") is put into the refine scene (if the scene held another model, it is moved to a folder beside it), and the pairing window opens with the pairs it suggests, marked **(proposal)**: a camera box gets a suggestion where the model scores one of the LiDAR boxes 0.5 or more; the others show no LiDAR id. If no base model was trained or chosen since the first cell was last run, the model already in the refine scene is used, or a dialog asks for the base model's folder. Pairs copied in by the optional cell "use pairs you already have for the refine data" are shown as saved pairs and get no suggestion.

A suggestion that is right needs no click. When one is wrong, click the camera box and then the right LiDAR box, or **— none —**. Then click **Apply this frame**: it saves the frame as it is shown, so a suggestion you left as it is counts as right.

Please apply a number of frames spread over the scene, with two pairs or more in each, and look at every camera box of a frame before you apply it. A camera box without a suggestion gets its LiDAR box from you when it has one: the fine-tuning in step 7 counts a camera box left without a LiDAR box, in a frame with a saved pair, as "no LiDAR box". **Apply all** saves every frame you opened in the window, as it was shown. Then close the window; if frames with a saved pair still have camera boxes without a decision, the cell lists them.

In [ ]:
flow.pair("refine")

## 7. Fine-tune the model

Run the next cell. The base model is trained further on the frames of the refine scene that hold saved pairs, for the number of epochs set in the first cell: their pairs, their **— none —** decisions and their camera boxes without a decision, which count as "no LiDAR box" too (the cell says how many). If the base model was neither trained nor chosen since the first cell was last run, a dialog asks for its folder. The base model itself is not changed; the result is a new model in a new folder `runs/customer-refine-<date>-<time>`. The cell names the base model it starts from.

**To fine-tune again with other epochs**, change the second number in the first cell and run that cell again. Then run, from top to bottom: step 1 with the same folder if Part 1 ran (step 9 needs its pairs; nothing is copied again and nothing is trained), step 5 with the same folder (it says "Already imported" and keeps every saved pair), the optional cell "use a base model you already have", in which you choose the base model used before (the `model` folder of step 4's run `runs/customer-base-<date>-<time>`, or the model you chose before), and this cell; step 6 can be skipped. The fine-tuning starts again from that base model, and the earlier fine-tuned model stays in its folder. It learns from every pair saved at that moment, the pairs applied in step 8 included, so the best moment to try other epochs is before you apply the frames of step 8: close that window without applying, and run the cells above. Step 8 suggests pairs only for camera boxes that have no decision yet.

In [ ]:
flow.train("refine")

## 8. Let the model pair the remaining frames

Run the next cell. The fine-tuned model is put into the refine scene; the model that was there is moved to a folder beside it, shown below. (If the fine-tuned model was neither trained nor chosen since the first cell was last run, a dialog first asks for its folder: the `model` folder shown below step 7.) The model then suggests a LiDAR box for every camera box of the refine scene that has no decision yet, where it scores one 0.5 or more; a camera box it finds none for keeps no decision. Two check images below the cell show its pairs: each pair has one colour, and the number after the LiDAR id is the model's score.

The window then opens with these pairs marked **(to confirm)**. They are predictions and are not saved yet: please go through the frames, correct what is wrong and click **Apply this frame**. **Apply all** saves every pair you did not change, including those in frames you have not looked at, so please look first. Close the window when you are done.

In [ ]:
flow.pair_remaining("refine")

# Homography

Step 9 writes the homography of the refine data from its saved pairs, and uses the base scene's pairs as well when Part 1 ran in this session. Please run it after Part 2; Part 1 is not needed for it.

## 9. Write the homography

Run the next cell after Part 2. The homography is fitted to the saved pairs of the refine scene of step 5, and of the base scene of step 1 when Part 1 ran in this session. A homography belongs to one camera position and one LiDAR table, so the base data has to come from the same camera position as the refine data: the cell says for each scene how many of its pairs agree with the homography, and warns when the pairs of one scene do not (the camera may have moved between the two recordings).

Part 1 is not needed: without a base scene, the refine scene's pairs are used alone, and then they have to cover the left, the middle and the right of the image. When they do not, the cell says so; base data of the same camera with its pairs (step 1 and the optional cell "use pairs you already have") then completes them. (If the notebook was restarted or the first cell was run again, please run, after the first cell, step 1 again if Part 1 ran and step 5 again, with the same folders: nothing is copied again, and without step 1 the base pairs are left out.) The homography is written to a new folder under `runs`:

- **`homography.json`**: the 3×3 matrix that turns `position_x, position_y` of a row of the delivered LiDAR table, as delivered, into the pixel of that object's centre, with the checks below.
- **`projection_inputs.py`**: the same values as Python assignments, ready for a projection script.
- **Projection images**: camera frames with a red dot for every LiDAR object and its id in green. Every dot should lie on its object.

Two numbers are printed as a check, counted over the saved pairs whose camera box is whole and not wide: how many agree with the homography within 25 px, and how many have the dot of their LiDAR object inside the paired camera box. A pair whose camera box is cut by the image edge, or wider than 8% of the image (an object close to the camera), takes no part in the fit or in these checks, because the centre of such a box is not the centre of the object. In the fit, a parked object counts once instead of once per frame, and the places are taken evenly over the image, so that the side with more traffic does not decide alone. All your pairs stay saved and are used for the training. If dots are off in one part of the image, or the cell says that one part has no support, please add pairs of moving objects there and run this cell again.

In [ ]:
flow.homography()